In [ ]:
import pickle
from pathlib import Path
import numpy as np
from tqdm import tqdm
import librosa
from sklearn.preprocessing import LabelEncoder
import shutil


In [ ]:
def create_melspectrogram(segmented_dir, melspectrogram_dir, override=False, n_mels=128):
    segmented_dir = Path(segmented_dir)
    melspectrogram_dir = Path(melspectrogram_dir)
    X = []
    y = []
    labels = [f.name for f in segmented_dir.iterdir() if f.is_dir()]

    if not segmented_dir.exists() or not any(segmented_dir.rglob('*.wav')):
        raise FileNotFoundError(
            f"Segmented audio files not found in {segmented_dir}. "
            "Run `clean.ipynb` first to generate segmented audio before extracting features."
        )

    if melspectrogram_dir.exists() and any(melspectrogram_dir.iterdir()):
        if override:
            shutil.rmtree(melspectrogram_dir)
            melspectrogram_dir.mkdir(parents=True)
        else:
            print('Melspectrogram directory not empty and override is False. Returning.')
            return
    
    melspectrogram_dir.mkdir(parents=True, exist_ok=True)

    tqdm._instances.clear()
    for label in tqdm(labels):
        for audio in (segmented_dir / label).iterdir():
            if audio.suffix == '.wav':
                signal, sr = librosa.load(audio, sr=None)
                melspectrogram = librosa.feature.melspectrogram(y=signal,
                                                              sr=sr,
                                                              n_mels=n_mels,
                                                              fmax=sr/2)
                melspectrogram_db = librosa.power_to_db(melspectrogram, ref=np.max)
                X.append(melspectrogram_db)
                y.append(label)

    X = np.array(X)
    X = np.expand_dims(X, axis=-1)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)

    np.save(melspectrogram_dir / 'X.npy', X)
    np.save(melspectrogram_dir / 'y.npy', np.array(y))

    with open('../../models/melspectrogram_label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)


In [ ]:
def create_mfcc(segmented_dir, mfcc_dir, override=False, n_mfcc=13, hop_length=512):
    segmented_dir = Path(segmented_dir)
    mfcc_dir = Path(mfcc_dir)
    X = []
    y = []
    labels = [f.name for f in segmented_dir.iterdir() if f.is_dir()]

    if mfcc_dir.exists() and any(mfcc_dir.iterdir()):
        if override:
            shutil.rmtree(mfcc_dir)
            mfcc_dir.mkdir(parents=True)
        else:
            print('MFCC directory not empty and override is False. Returning.')
            return
        
    mfcc_dir.mkdir(parents=True, exist_ok=True)

    tqdm._instances.clear()
    for label in tqdm(labels):
        for audio in (segmented_dir / label).iterdir():
            if audio.suffix == '.wav':
                signal, sr = librosa.load(audio, sr=None)
                n_fft = min(2048, len(signal))
                mfcc = librosa.feature.mfcc(y=signal,
                                            sr=sr,
                                            n_mfcc=n_mfcc,
                                            n_fft=n_fft,
                                            hop_length=hop_length)
                X.append(mfcc)
                y.append(label)

    X = np.array(X)
    X = np.expand_dims(X, axis=-1)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)

    np.save(mfcc_dir / 'X.npy', X)
    np.save(mfcc_dir / 'y.npy', np.array(y))

    with open('../../models/mfcc_label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)


In [ ]:
def create_mfcc_stats(clean_dir, mfcc_stats_dir, override=False, n_mfcc=13, hop_length=512):
    clean_dir = Path(clean_dir)
    mfcc_stats_dir = Path(mfcc_stats_dir)

    if not clean_dir.exists() or not any(clean_dir.rglob('*.wav')):
        raise FileNotFoundError(
            f"Clean audio files not found in {clean_dir}. "
            "Run `clean.ipynb` first and verify `code/IRMAS/data/clean/` contains files before extracting features."
        )

    if mfcc_stats_dir.exists() and any(mfcc_stats_dir.iterdir()):
        if override:
            shutil.rmtree(mfcc_stats_dir)
            mfcc_stats_dir.mkdir(parents=True)
        else:
            print('MFCC stats directory not empty and override is False. Returning.')
            return
        
    mfcc_stats_dir.mkdir(parents=True, exist_ok=True)

    X = []
    y = []
    tqdm._instances.clear()
    labels = [f.name for f in clean_dir.iterdir() if f.is_dir()]
    for label in tqdm(labels):
        for audio in (clean_dir / label).iterdir():
            if audio.suffix == '.wav':
                signal, sr = librosa.load(audio, sr=None)
                n_fft = min(2048, len(signal))
                mfcc = librosa.feature.mfcc(y=signal,
                                            sr=sr,
                                            n_mfcc=n_mfcc,
                                            n_fft=n_fft,
                                            hop_length=hop_length)
                mfcc_mean = np.mean(mfcc, axis=1)
                mfcc_std = np.std(mfcc, axis=1)
                X.append(np.concatenate([mfcc_mean, mfcc_std]))
                y.append(label)

    np.save(mfcc_stats_dir / 'X.npy', np.array(X))
    np.save(mfcc_stats_dir / 'y.npy', np.array(y))


Extract features from files created by `clean.ipynb`


In [10]:
create_melspectrogram('../../data/segmented', '../../features/melspectrograms', override=True, n_mels=128)
create_mfcc('../../data/segmented', '../../features/mfcc', override=True, n_mfcc=13, hop_length=512)
create_mfcc_stats('../../data/clean', '../../features/mfcc_stats', override=True, n_mfcc=13, hop_length=512)


100%|██████████| 11/11 [00:50<00:00,  4.56s/it]
